# Visualisation des inférences (mode infer_from_tiles)

Ce notebook reproduit le pipeline de `infer_from_tiles.py` pour une fenêtre et une zone MGRS-C précises,
avec visualisation interactive RGB / NIR-R-G en mode **aléatoire** et **consécutif**.

In [19]:
import json
import warnings
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output
from omegaconf import OmegaConf

from dataloader_CIRCA.datasets.dataset_from_files import Dataset_from_files
from dataloader_CIRCA.tools.data_processor import SentinelDataProcessor
from lib import config_utils, visutils
from lib.eval_tools import Imputation

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline

## 1. Paramètres

In [22]:
# ── Zone / fenêtre ─────────────────────────────────────────────────────────────
MGRSC = "31TGJ_row-3_col-2"
WINDOW = (300, 600, 128, 128)  # (col_off, row_off, width, height)

# ── Données ────────────────────────────────────────────────────────────────────
store_dai = Path("/mnt/stores/store_dai")
data_optique = store_dai / "projets/pac/3str/EXP_2/Data_Raster/optique_dataset"
data_radar   = store_dai / "projets/pac/3str/EXP_2/Data_Raster/radar_dataset_v4"
data_masks_aleatoire  = store_dai / "projets/pac/3str/EXP_2/Data_Raster/test_v3/aleatoire"
data_masks_consecutif = store_dai / "projets/pac/3str/EXP_2/Data_Raster/test_v3/consecutif"

# ── Modèle (v3_combined mix_closest) ───────────────────────────────────────────
path_ckpt_config = store_dai / "tmp/speillet/cloud_reconstruction_results/U-TILISE/" \
    "v3_mix_closest_random_clouds_combined/2026-03-26_16-17/config.yaml"
path_ckpt_pth = store_dai / "tmp/speillet/cloud_reconstruction_results/U-TILISE/" \
    "v3_mix_closest_random_clouds_combined/2026-03-26_16-17/checkpoints/Model_best.pth"

# ── Config d'inférence ─────────────────────────────────────────────────────────
path_inference_config = Path("./configs/config_run_eval.yaml")
USE_SAR = "mix_closest"
FILL_VALUE = 1.0
IMAGE_SIZE = [128, 128]

## 2. Fonctions utilitaires

In [23]:
def _to_cpu(x):
    """Copie récursive de tenseurs en CPU."""
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().clone()
    elif isinstance(x, dict):
        return {k: _to_cpu(v) for k, v in x.items()}
    elif isinstance(x, (list, tuple)):
        return type(x)(_to_cpu(v) for v in x)
    return x


def sample_to_batch(sample: dict) -> dict:
    batch = {}
    for k, v in sample.items():
        if isinstance(v, torch.Tensor):
            batch[k] = v.unsqueeze(dim=0)
        else:
            batch[k] = v
    return batch


def _draw_grid(images, ncols, column_labels, row_labels, ax_array, fig, BRIGHTNESS_FACTOR, show_col_labels=True):
    """Dessine la grille Target / Inputs / Predictions pour un sous-ensemble de timesteps."""
    images = visutils.apply_brightness_factor(images, factor=BRIGHTNESS_FACTOR)
    for idx, ax in enumerate(ax_array.flat):
        ax.imshow(images[idx])
        ax.axis('off')
        rect = mpatches.Rectangle((0, 0), 1, 1,
                                  transform=ax.transAxes,
                                  fill=False, edgecolor='black', linewidth=1)
        ax.add_patch(rect)
        if show_col_labels and idx < ncols:
            ax.set_title(column_labels[idx], fontsize=9, fontweight='bold', rotation=45, ha='left')
    for i, label in enumerate(row_labels):
        ax_array[i, 0].annotate(label, xy=(-0.1, 0.5), xycoords='axes fraction',
                                fontsize=10, fontweight='bold',
                                ha='right', va='center', rotation=90)


def plot_seq_RGB_NIR_interactive(targets, inputs, preds, n_visible=8, title="", dates=None):
    """Slider interactif RGB + NIR-R-G. dates: liste de strings YYYY-MM-DD pour les labels."""
    targets = _to_cpu(targets)
    inputs  = _to_cpu(inputs)
    preds   = _to_cpu(preds)

    target_rgb = targets[:, [2, 1, 0], :, :].swapaxes(1, 3).swapaxes(1, 2)
    inputs_rgb = inputs[:, [2, 1, 0], :, :].swapaxes(1, 3).swapaxes(1, 2)
    preds_rgb  = preds[:, [2, 1, 0], :, :].swapaxes(1, 3).swapaxes(1, 2)

    target_nir = targets[:, [6, 2, 1], :, :].swapaxes(1, 3).swapaxes(1, 2)
    inputs_nir = inputs[:, [6, 2, 1], :, :].swapaxes(1, 3).swapaxes(1, 2)
    preds_nir  = preds[:, [6, 2, 1], :, :].swapaxes(1, 3).swapaxes(1, 2)

    T = target_rgb.shape[0]
    n_visible = min(n_visible, T)
    if dates is None:
        dates = [f't{i}' for i in range(T)]

    out = widgets.Output()
    slider = widgets.IntSlider(
        value=0, min=0, max=max(T - n_visible, 0), step=1,
        description='t_start :',
        continuous_update=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='80%'),
    )
    label = widgets.Label(
        value=f'{title}  —  Série totale : {T} dates  |  Fenêtre : {n_visible}'
    )

    def _update(_change=None):
        t0 = slider.value
        idx = slice(t0, t0 + n_visible)
        col_labels = dates[t0:t0 + n_visible]
        imgs_rgb = torch.concatenate([target_rgb[idx], inputs_rgb[idx], preds_rgb[idx]], dim=0)
        imgs_nir = torch.concatenate([target_nir[idx], inputs_nir[idx], preds_nir[idx]], dim=0)
        row_labels = ['Target', 'Inputs', 'Predictions']
        with out:
            clear_output(wait=True)
            fig, axes = plt.subplots(nrows=6, ncols=n_visible,
                                     figsize=(2.2 * n_visible, 13))
            _draw_grid(imgs_rgb, n_visible, col_labels, row_labels,
                       axes[:3], fig, BRIGHTNESS_FACTOR=3, show_col_labels=True)
            _draw_grid(imgs_nir, n_visible, col_labels, row_labels,
                       axes[3:], fig, BRIGHTNESS_FACTOR=2, show_col_labels=False)
            fig.text(0.02, 0.78, 'RGB', fontsize=14, fontweight='bold',
                     rotation=90, va='center')
            fig.text(0.02, 0.35, 'NIR-R-G', fontsize=14, fontweight='bold',
                     rotation=90, va='center')
            plt.subplots_adjust(left=0.08, top=0.92, wspace=0.05, hspace=0.15)
            plt.show()

    slider.observe(_update, names='value')
    display(widgets.VBox([label, slider, out]))
    _update()

## 3. Chargement du modèle d'imputation

In [14]:
# On crée un Dataset temporaire juste pour obtenir num_channels
ds_tmp = Dataset_from_files(
    mgrsc=MGRSC,
    data_optique=data_optique,
    data_radar=data_radar,
    image_size=IMAGE_SIZE,
    overlap=0,
    fill_value=FILL_VALUE,
    mask_type='original_masks',
    use_sar=USE_SAR,
    keep_all_dates=True,
)

imputation = Imputation(
    config_file_train=str(path_inference_config),
    method="utilise",
    checkpoint=str(path_ckpt_pth),
    config_file_test=str(path_ckpt_config),
    num_channels=ds_tmp.num_channels,
)
del ds_tmp
print("Modèle chargé.")

mgrs:   0%|          | 0/35 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/2 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/4 [00:00<?, ?it/s]

...

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/1 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/86 [00:00<?, ?it/s]

Checkpoint '/mnt/stores/store_dai/tmp/speillet/cloud_reconstruction_results/U-TILISE/v3_mix_closest_random_clouds_combined/2026-03-26_16-17/checkpoints/Model_best.pth' loaded.
Chosen epoch: 116

Modèle chargé.


## 4. Mode aléatoire (random_fully_masked)

On masque synthétiquement des dates claires choisies aléatoirement (+150 sur les masques).
Le modèle doit reconstruire ces dates masquées.

In [15]:
ds_rfm = Dataset_from_files(
    mgrsc=MGRSC,
    data_optique=data_optique,
    data_radar=data_radar,
    image_size=IMAGE_SIZE,
    overlap=0,
    fill_value=FILL_VALUE,
    mask_type='fully_masked',
    data_masks=data_masks_aleatoire,
    use_sar=USE_SAR,
    keep_all_dates=False,
)
print(f"Dataset rfm: {len(ds_rfm)} patches, {ds_rfm.num_channels} channels")

mgrs:   0%|          | 0/35 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/2 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/4 [00:00<?, ?it/s]

...

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/1 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/86 [00:00<?, ?it/s]

Dataset rfm: 324 patches, 14 channels


In [16]:
# Inférence sur la fenêtre choisie via get_item_from_mgrsc
x, y, w, h = WINDOW
sample_rfm = ds_rfm.get_item_from_mgrsc(
    mgrsc=MGRSC,
    x=x, y=y, width=w, height=h,
)
batch_rfm = sample_to_batch(sample_rfm)
batch_rfm, y_pred_rfm = imputation.impute_sample(batch_rfm)

print(f"Dates S2 : {len(sample_rfm['S2_dates'])}")
print(f"Prédiction shape : {y_pred_rfm.shape}")

Dates S2 : 48
Prédiction shape : torch.Size([1, 48, 10, 128, 128])


In [ ]:
plot_seq_RGB_NIR_interactive(
    targets=batch_rfm["y"][0],
    inputs=batch_rfm["x"][0],
    preds=y_pred_rfm[0],
    n_visible=8,
    title=f"Mode ALÉATOIRE — {MGRSC} — fenêtre {WINDOW}",
    dates=sample_rfm["S2_dates"],
)

## 5. Mode consécutif (consecutive_fully_masked)

On masque synthétiquement un bloc consécutif de dates claires (+150 sur les masques).
Le modèle doit reconstruire cette lacune temporelle.

In [25]:
ds_cfm = Dataset_from_files(
    mgrsc=MGRSC,
    data_optique=data_optique,
    data_radar=data_radar,
    image_size=IMAGE_SIZE,
    overlap=0,
    fill_value=FILL_VALUE,
    mask_type='fully_masked',
    data_masks=data_masks_consecutif,
    use_sar=USE_SAR,
    keep_all_dates=False,
)
print(f"Dataset cfm: {len(ds_cfm)} patches, {ds_cfm.num_channels} channels")

mgrs:   0%|          | 0/35 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/2 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/4 [00:00<?, ?it/s]

...

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/1 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/86 [00:00<?, ?it/s]

Dataset cfm: 324 patches, 14 channels


In [26]:
x, y, w, h = WINDOW
sample_cfm = ds_cfm.get_item_from_mgrsc(
    mgrsc=MGRSC,
    x=x, y=y, width=w, height=h,
)
batch_cfm = sample_to_batch(sample_cfm)
batch_cfm, y_pred_cfm = imputation.impute_sample(batch_cfm)

print(f"Dates S2 : {len(sample_cfm['S2_dates'])}")
print(f"Prédiction shape : {y_pred_cfm.shape}")

Dates S2 : 50
Prédiction shape : torch.Size([1, 50, 10, 128, 128])


In [ ]:
plot_seq_RGB_NIR_interactive(
    targets=batch_cfm["y"][0],
    inputs=batch_cfm["x"][0],
    preds=y_pred_cfm[0],
    n_visible=8,
    title=f"Mode CONSÉCUTIF — {MGRSC} — fenêtre {WINDOW}",
    dates=sample_cfm["S2_dates"],
)

## 6. Mode produit (original_masks — masques de nuages réels)

Pas de masquage synthétique : le modèle reconstruit directement les pixels nuageux
à partir des masques d'origine. Mode utilisé pour produire les inférences finales.

In [28]:
ds_prod = Dataset_from_files(
    mgrsc=MGRSC,
    data_optique=data_optique,
    data_radar=data_radar,
    image_size=IMAGE_SIZE,
    overlap=0,
    fill_value=FILL_VALUE,
    mask_type='original_masks',
    use_sar=USE_SAR,
    keep_all_dates=True,
)
print(f"Dataset produit: {len(ds_prod)} patches, {ds_prod.num_channels} channels")

mgrs:   0%|          | 0/35 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/2 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/4 [00:00<?, ?it/s]

...

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/1 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/86 [00:00<?, ?it/s]

Dataset produit: 324 patches, 14 channels


In [29]:
x, y, w, h = WINDOW
sample_prod = ds_prod.get_item_from_mgrsc(
    mgrsc=MGRSC,
    x=x, y=y, width=w, height=h,
)
batch_prod = sample_to_batch(sample_prod)
batch_prod, y_pred_prod = imputation.impute_sample(batch_prod)

print(f"Dates S2 : {len(sample_prod['S2_dates'])}")
print(f"Prédiction shape : {y_pred_prod.shape}")

Dates S2 : 77
Prédiction shape : torch.Size([1, 77, 10, 128, 128])


In [ ]:
plot_seq_RGB_NIR_interactive(
    targets=batch_prod["y"][0],
    inputs=batch_prod["x"][0],
    preds=y_pred_prod[0],
    n_visible=8,
    title=f"Mode PRODUIT (original_masks) — {MGRSC} — fenêtre {WINDOW}",
    dates=sample_prod["S2_dates"],
)

### Mode customs MGRSC 1


In [ ]:
# On utilise le dataset "produit" (original_masks) déjà créé pour la 1ère zone
# pour lister les dates disponibles
ds_custom = Dataset_from_files(
    mgrsc=MGRSC,
    data_optique=data_optique,
    data_radar=data_radar,
    image_size=IMAGE_SIZE,
    overlap=0,
    fill_value=FILL_VALUE,
    mask_type='original_masks',
    use_sar=USE_SAR,
    keep_all_dates=True,
)

# Récupérer les dates et masques de nuages pour cette fenêtre
sample_dates = ds_custom.get_item_from_mgrsc(
    mgrsc=MGRSC, x=WINDOW[0], y=WINDOW[1], width=WINDOW[2], height=WINDOW[3],
)

# Calculer le % de nuages par date (cloud_mask: 1=nuageux, 0=clair)
cloud_pct = sample_dates["cloud_mask"].squeeze(1).mean(dim=(1, 2)) * 100  # (T,)
CLOUD_THRESHOLD = 10  # % au-dessus duquel on considère la date comme nuageuse

clear_idx = [i for i, p in enumerate(cloud_pct) if p < CLOUD_THRESHOLD]
cloudy_idx = [i for i, p in enumerate(cloud_pct) if p >= CLOUD_THRESHOLD]

print(f"Seuil nuages : {CLOUD_THRESHOLD}%\n")
print("Dates NON-NUAGEUSES (clear) — sélectionnables pour masquage :")
for i in clear_idx:
    print(f"  [{i:2d}] {sample_dates['S2_dates'][i]}  ({cloud_pct[i]:.1f}%)")
print(f"\nDates NUAGEUSES (déjà masquées par le modèle) :")
for i in cloudy_idx:
    print(f"  [{i:2d}] {sample_dates['S2_dates'][i]}  ({cloud_pct[i]:.1f}%)")
print(f"\nTotal : {len(clear_idx)} claires, {len(cloudy_idx)} nuageuses sur {len(cloud_pct)} dates")

In [ ]:
# ── Choisir les indices des dates CLAIRES à masquer ────────────────────────────
# Ne choisir que parmi les indices affichés comme "NON-NUAGEUSES" ci-dessus.
# Exemples : masquer la 2ème, 4ème et 6ème date claire
DATES_TO_MASK = [clear_idx[2], clear_idx[4], clear_idx[6], clear_idx[39], clear_idx[41], clear_idx[43]]  # ex: indices 39, 41, 43 dans la liste des dates claires
# Ou directement par indice absolu, ex: DATES_TO_MASK = [3, 5, 8]

print(f"Indices masqués : {DATES_TO_MASK}")
print(f"Dates masquées  : {[sample_dates['S2_dates'][i] for i in DATES_TO_MASK]}")
print(f"(les dates nuageuses seront retirées de la séquence)")

sample_custom = ds_custom.get_item_from_mgrsc(
    mgrsc=MGRSC,
    x=WINDOW[0], y=WINDOW[1], width=WINDOW[2], height=WINDOW[3],
    dates_to_mask=DATES_TO_MASK,
)
batch_custom = sample_to_batch(sample_custom)
batch_custom, y_pred_custom = imputation.impute_sample(batch_custom)

T_out = len(sample_custom["S2_dates"])
print(f"\nSéquence résultante : {T_out} dates (claires uniquement)")
print(f"Dates dans la séquence : {sample_custom['S2_dates']}")
print(f"Prédiction shape : {y_pred_custom.shape}")


plot_seq_RGB_NIR_interactive(
    targets=batch_custom["y"][0],
    inputs=batch_custom["x"][0],
    preds=y_pred_custom[0],
    n_visible=8,
    title=f"Masquage MANUEL dates {DATES_TO_MASK} — {MGRSC} — fenêtre {WINDOW}",
    dates=sample_custom["S2_dates"],
)

---

## Zone 31TGJ_row-3_col-2 — Fenêtre (668, 2000, 128, 128)

Même pipeline pour une seconde zone / fenêtre.

In [26]:
MGRSC_2 = "31TGJ_row-3_col-2"
WINDOW_2 = (668, 2000, 128, 128)

### Mode aléatoire (random_fully_masked)

In [32]:
ds_rfm_2 = Dataset_from_files(
    mgrsc=MGRSC_2,
    data_optique=data_optique,
    data_radar=data_radar,
    image_size=[128, 128],
    overlap=0,
    fill_value=FILL_VALUE,
    mask_type='fully_masked',
    data_masks=data_masks_aleatoire,
    use_sar=USE_SAR,
    keep_all_dates=False,
)
print(f"Dataset rfm_2: {len(ds_rfm_2)} patches, {ds_rfm_2.num_channels} channels")

mgrs:   0%|          | 0/35 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/2 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/4 [00:00<?, ?it/s]

...

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/1 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/86 [00:00<?, ?it/s]

Dataset rfm_2: 324 patches, 14 channels


In [33]:
x2, y2, w2, h2 = WINDOW_2
sample_rfm_2 = ds_rfm_2.get_item_from_mgrsc(
    mgrsc=MGRSC_2,
    x=x2, y=y2, width=w2, height=h2,
)
batch_rfm_2 = sample_to_batch(sample_rfm_2)
batch_rfm_2, y_pred_rfm_2 = imputation.impute_sample(batch_rfm_2)

print(f"Dates S2 : {len(sample_rfm_2['S2_dates'])}")
print(f"Prédiction shape : {y_pred_rfm_2.shape}")

Dates S2 : 11
Prédiction shape : torch.Size([1, 11, 10, 128, 128])


In [ ]:
plot_seq_RGB_NIR_interactive(
    targets=batch_rfm_2["y"][0],
    inputs=batch_rfm_2["x"][0],
    preds=y_pred_rfm_2[0],
    n_visible=8,
    title=f"Mode ALÉATOIRE — {MGRSC_2} — fenêtre {WINDOW_2}",
    dates=sample_rfm_2["S2_dates"],
)

### Mode consécutif (consecutive_fully_masked)

In [35]:
ds_cfm_2 = Dataset_from_files(
    mgrsc=MGRSC_2,
    data_optique=data_optique,
    data_radar=data_radar,
    image_size=[128, 128],
    overlap=0,
    fill_value=FILL_VALUE,
    mask_type='fully_masked',
    data_masks=data_masks_consecutif,
    use_sar=USE_SAR,
    keep_all_dates=False,
)
print(f"Dataset cfm_2: {len(ds_cfm_2)} patches, {ds_cfm_2.num_channels} channels")

mgrs:   0%|          | 0/35 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/2 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/4 [00:00<?, ?it/s]

...

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/1 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/86 [00:00<?, ?it/s]

Dataset cfm_2: 324 patches, 14 channels


In [36]:
x2, y2, w2, h2 = WINDOW_2
sample_cfm_2 = ds_cfm_2.get_item_from_mgrsc(
    mgrsc=MGRSC_2,
    x=x2, y=y2, width=w2, height=h2,
)
batch_cfm_2 = sample_to_batch(sample_cfm_2)
batch_cfm_2, y_pred_cfm_2 = imputation.impute_sample(batch_cfm_2)

print(f"Dates S2 : {len(sample_cfm_2['S2_dates'])}")
print(f"Prédiction shape : {y_pred_cfm_2.shape}")

Dates S2 : 29
Prédiction shape : torch.Size([1, 29, 10, 128, 128])


In [ ]:
plot_seq_RGB_NIR_interactive(
    targets=batch_cfm_2["y"][0],
    inputs=batch_cfm_2["x"][0],
    preds=y_pred_cfm_2[0],
    n_visible=8,
    title=f"Mode CONSÉCUTIF — {MGRSC_2} — fenêtre {WINDOW_2}",
    dates=sample_cfm_2["S2_dates"],
)

### Mode produit (original_masks)

In [38]:
ds_prod_2 = Dataset_from_files(
    mgrsc=MGRSC_2,
    data_optique=data_optique,
    data_radar=data_radar,
    image_size=[128, 128],
    overlap=0,
    fill_value=FILL_VALUE,
    mask_type='original_masks',
    use_sar=USE_SAR,
    keep_all_dates=True,
)
print(f"Dataset prod_2: {len(ds_prod_2)} patches, {ds_prod_2.num_channels} channels")

mgrs:   0%|          | 0/35 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/2 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/4 [00:00<?, ?it/s]

...

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/1 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/86 [00:00<?, ?it/s]

Dataset prod_2: 324 patches, 14 channels


In [39]:
x2, y2, w2, h2 = WINDOW_2
sample_prod_2 = ds_prod_2.get_item_from_mgrsc(
    mgrsc=MGRSC_2,
    x=x2, y=y2, width=w2, height=h2,
)
batch_prod_2 = sample_to_batch(sample_prod_2)
batch_prod_2, y_pred_prod_2 = imputation.impute_sample(batch_prod_2)

print(f"Dates S2 : {len(sample_prod_2['S2_dates'])}")
print(f"Prédiction shape : {y_pred_prod_2.shape}")

Dates S2 : 77
Prédiction shape : torch.Size([1, 77, 10, 128, 128])


In [ ]:
plot_seq_RGB_NIR_interactive(
    targets=batch_prod_2["y"][0],
    inputs=batch_prod_2["x"][0],
    preds=y_pred_prod_2[0],
    n_visible=8,
    title=f"Mode PRODUIT (original_masks) — {MGRSC_2} — fenêtre {WINDOW_2}",
    dates=sample_prod_2["S2_dates"],
)

### Mode dates customs MGRSC2

In [28]:
# pour lister les dates disponibles
ds_custom_2 = Dataset_from_files(
    mgrsc=MGRSC_2,
    data_optique=data_optique,
    data_radar=data_radar,
    image_size=IMAGE_SIZE,
    overlap=0,
    fill_value=FILL_VALUE,
    mask_type='original_masks',
    use_sar=USE_SAR,
    keep_all_dates=True,
)

# Récupérer les dates et masques de nuages pour cette fenêtre
sample_dates = ds_custom_2.get_item_from_mgrsc(
    mgrsc=MGRSC_2, x=WINDOW_2[0], y=WINDOW_2[1], width=WINDOW_2[2], height=WINDOW_2[3],
)

# Calculer le % de nuages par date (cloud_mask: 1=nuageux, 0=clair)
cloud_pct = sample_dates["cloud_mask"].squeeze(1).mean(dim=(1, 2)) * 100  # (T,)
CLOUD_THRESHOLD = 10  # % au-dessus duquel on considère la date comme nuageuse

clear_idx = [i for i, p in enumerate(cloud_pct) if p < CLOUD_THRESHOLD]
cloudy_idx = [i for i, p in enumerate(cloud_pct) if p >= CLOUD_THRESHOLD]

print(f"Seuil nuages : {CLOUD_THRESHOLD}%\n")
print("Dates NON-NUAGEUSES (clear) — sélectionnables pour masquage :")
for i in clear_idx:
    print(f"  [{i:2d}] {sample_dates['S2_dates'][i]}  ({cloud_pct[i]:.1f}%)")
print(f"\nDates NUAGEUSES (déjà masquées par le modèle) :")
for i in cloudy_idx:
    print(f"  [{i:2d}] {sample_dates['S2_dates'][i]}  ({cloud_pct[i]:.1f}%)")
print(f"\nTotal : {len(clear_idx)} claires, {len(cloudy_idx)} nuageuses sur {len(cloud_pct)} dates")

mgrs:   0%|          | 0/35 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/2 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/4 [00:00<?, ?it/s]

...

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/1 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/86 [00:00<?, ?it/s]

Seuil nuages : 10%

Dates NON-NUAGEUSES (clear) — sélectionnables pour masquage :
  [ 0] 2022-09-01  (1.4%)
  [ 2] 2022-09-11  (0.2%)
  [ 3] 2022-09-16  (0.1%)
  [ 4] 2022-09-21  (0.0%)
  [ 5] 2022-09-26  (0.0%)
  [ 7] 2022-10-06  (0.2%)
  [ 8] 2022-10-11  (0.1%)
  [11] 2022-10-26  (0.0%)
  [13] 2022-11-05  (0.0%)
  [15] 2022-11-20  (0.0%)
  [16] 2022-11-25  (1.6%)
  [17] 2022-11-30  (0.2%)
  [18] 2022-12-05  (0.0%)
  [23] 2023-01-04  (0.3%)
  [24] 2023-01-09  (0.0%)
  [26] 2023-01-19  (0.0%)
  [27] 2023-01-24  (0.8%)
  [28] 2023-01-29  (0.0%)
  [29] 2023-02-03  (0.0%)
  [30] 2023-02-08  (0.0%)
  [31] 2023-02-13  (0.0%)
  [32] 2023-02-18  (0.0%)
  [35] 2023-03-05  (0.0%)
  [36] 2023-03-10  (0.2%)
  [37] 2023-03-15  (0.0%)
  [39] 2023-03-25  (6.3%)
  [40] 2023-03-30  (0.0%)
  [44] 2023-04-19  (0.5%)
  [45] 2023-04-24  (1.9%)
  [46] 2023-04-29  (1.5%)
  [48] 2023-05-09  (7.5%)
  [49] 2023-05-14  (0.0%)
  [51] 2023-05-24  (0.0%)
  [53] 2023-06-03  (0.2%)
  [54] 2023-06-08  (0.0%)
  [56] 2

In [35]:
# ── Choisir les indices des dates CLAIRES à masquer ────────────────────────────
# Ne choisir que parmi les indices affichés comme "NON-NUAGEUSES" ci-dessus.
# Exemples : masquer la 2ème, 4ème et 6ème date claire
DATES_TO_MASK = [clear_idx[35], clear_idx[37]]  # ex: indices 39, 41, 43 dans la liste des dates claires
# Ou directement par indice absolu, ex: DATES_TO_MASK = [3, 5, 8]

print(f"Indices masqués : {DATES_TO_MASK}")
print(f"Dates masquées  : {[sample_dates['S2_dates'][i] for i in DATES_TO_MASK]}")
print(f"(les dates nuageuses seront retirées de la séquence)")

sample_custom = ds_custom_2.get_item_from_mgrsc(
    mgrsc=MGRSC_2,
    x=WINDOW_2[0], y=WINDOW_2[1], width=WINDOW_2[2], height=WINDOW_2[3],
    dates_to_mask=DATES_TO_MASK,
)
batch_custom = sample_to_batch(sample_custom)
batch_custom, y_pred_custom = imputation.impute_sample(batch_custom)

T_out = len(sample_custom["S2_dates"])
print(f"\nSéquence résultante : {T_out} dates (claires uniquement)")
print(f"Dates dans la séquence : {sample_custom['S2_dates']}")
print(f"Prédiction shape : {y_pred_custom.shape}")

Indices masqués : [56, 58]
Dates masquées  : ['2023-06-18', '2023-06-28']
(les dates nuageuses seront retirées de la séquence)

Séquence résultante : 49 dates (claires uniquement)
Dates dans la séquence : ['2022-09-01', '2022-09-11', '2022-09-16', '2022-09-21', '2022-09-26', '2022-10-06', '2022-10-11', '2022-10-26', '2022-11-05', '2022-11-20', '2022-11-25', '2022-11-30', '2022-12-05', '2023-01-04', '2023-01-09', '2023-01-19', '2023-01-24', '2023-01-29', '2023-02-03', '2023-02-08', '2023-02-13', '2023-02-18', '2023-03-05', '2023-03-10', '2023-03-15', '2023-03-30', '2023-04-19', '2023-04-24', '2023-04-29', '2023-05-14', '2023-05-24', '2023-06-03', '2023-06-08', '2023-06-18', '2023-06-23', '2023-06-28', '2023-07-03', '2023-07-08', '2023-07-13', '2023-07-23', '2023-07-28', '2023-08-02', '2023-08-07', '2023-08-12', '2023-08-22', '2023-09-01', '2023-09-06', '2023-09-11', '2023-09-26']
Prédiction shape : torch.Size([1, 49, 10, 128, 128])


In [36]:
plot_seq_RGB_NIR_interactive(
    targets=batch_custom["y"][0],
    inputs=batch_custom["x"][0],
    preds=y_pred_custom[0],
    n_visible=8,
    title=f"Mode CUSTOM — {MGRSC_2} — fenêtre {WINDOW_2}",
    dates=sample_custom["S2_dates"],
)

---

## Masquage manuel de dates choisies (`dates_to_mask`)

Le paramètre `dates_to_mask` de `get_item_from_mgrsc()` :

1. **Filtre les dates nuageuses** de la séquence (comme le mode `fully_masked`)
2. Ne garde que les **dates claires** dans l'input
3. Parmi celles-ci, **masque entièrement** les dates choisies &#8594; le modèle doit les reconstruire

Les indices sont les indices **absolus** dans la série temporelle originale (affichés ci-dessous).

In [17]:
# On utilise le dataset "produit" (original_masks) déjà créé pour la 1ère zone
# pour lister les dates disponibles
ds_custom = Dataset_from_files(
    mgrsc=MGRSC,
    data_optique=data_optique,
    data_radar=data_radar,
    image_size=IMAGE_SIZE,
    overlap=0,
    fill_value=FILL_VALUE,
    mask_type='original_masks',
    use_sar=USE_SAR,
    keep_all_dates=True,
)

# Récupérer les dates et masques de nuages pour cette fenêtre
sample_dates = ds_custom.get_item_from_mgrsc(
    mgrsc=MGRSC, x=WINDOW[0], y=WINDOW[1], width=WINDOW[2], height=WINDOW[3],
)

# Calculer le % de nuages par date (cloud_mask: 1=nuageux, 0=clair)
cloud_pct = sample_dates["cloud_mask"].squeeze(1).mean(dim=(1, 2)) * 100  # (T,)
CLOUD_THRESHOLD = 10  # % au-dessus duquel on considère la date comme nuageuse

clear_idx = [i for i, p in enumerate(cloud_pct) if p < CLOUD_THRESHOLD]
cloudy_idx = [i for i, p in enumerate(cloud_pct) if p >= CLOUD_THRESHOLD]

print(f"Seuil nuages : {CLOUD_THRESHOLD}%\n")
print("Dates NON-NUAGEUSES (clear) — sélectionnables pour masquage :")
for i in clear_idx:
    print(f"  [{i:2d}] {sample_dates['S2_dates'][i]}  ({cloud_pct[i]:.1f}%)")
print(f"\nDates NUAGEUSES (déjà masquées par le modèle) :")
for i in cloudy_idx:
    print(f"  [{i:2d}] {sample_dates['S2_dates'][i]}  ({cloud_pct[i]:.1f}%)")
print(f"\nTotal : {len(clear_idx)} claires, {len(cloudy_idx)} nuageuses sur {len(cloud_pct)} dates")

mgrs:   0%|          | 0/35 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/2 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/4 [00:00<?, ?it/s]

...

mgrs25:   0%|          | 0/3 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/1 [00:00<?, ?it/s]

mgrs25:   0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/86 [00:00<?, ?it/s]

Seuil nuages : 10%

Dates NON-NUAGEUSES (clear) — sélectionnables pour masquage :
  [ 0] 2022-09-01  (0.0%)
  [ 2] 2022-09-11  (0.0%)
  [ 3] 2022-09-16  (0.0%)
  [ 4] 2022-09-21  (0.3%)
  [ 5] 2022-09-26  (0.0%)
  [ 7] 2022-10-06  (0.0%)
  [ 8] 2022-10-11  (3.9%)
  [ 9] 2022-10-16  (9.4%)
  [11] 2022-10-26  (0.0%)
  [12] 2022-10-31  (2.1%)
  [13] 2022-11-05  (0.0%)
  [15] 2022-11-20  (0.2%)
  [17] 2022-11-30  (0.8%)
  [18] 2022-12-05  (0.4%)
  [22] 2022-12-25  (1.4%)
  [23] 2023-01-04  (0.1%)
  [26] 2023-01-19  (0.0%)
  [28] 2023-01-29  (0.0%)
  [29] 2023-02-03  (0.0%)
  [30] 2023-02-08  (0.0%)
  [31] 2023-02-13  (0.0%)
  [32] 2023-02-18  (0.0%)
  [35] 2023-03-05  (0.0%)
  [36] 2023-03-10  (8.1%)
  [37] 2023-03-15  (1.6%)
  [38] 2023-03-20  (0.0%)
  [39] 2023-03-25  (0.0%)
  [40] 2023-03-30  (0.0%)
  [43] 2023-04-14  (0.2%)
  [44] 2023-04-19  (0.0%)
  [46] 2023-04-29  (6.8%)
  [47] 2023-05-04  (6.3%)
  [49] 2023-05-14  (1.5%)
  [51] 2023-05-24  (0.4%)
  [53] 2023-06-03  (0.1%)
  [54] 2

In [ ]:
# ── Choisir les indices des dates CLAIRES à masquer ────────────────────────────
# Ne choisir que parmi les indices affichés comme "NON-NUAGEUSES" ci-dessus.
# Exemples : masquer la 2ème, 4ème et 6ème date claire
DATES_TO_MASK = [clear_idx[2], clear_idx[4], clear_idx[6], clear_idx[39], clear_idx[41], clear_idx[43]]  # ex: indices 39, 41, 43 dans la liste des dates claires
# Ou directement par indice absolu, ex: DATES_TO_MASK = [3, 5, 8]

print(f"Indices masqués : {DATES_TO_MASK}")
print(f"Dates masquées  : {[sample_dates['S2_dates'][i] for i in DATES_TO_MASK]}")
print(f"(les dates nuageuses seront retirées de la séquence)")

sample_custom = ds_custom.get_item_from_mgrsc(
    mgrsc=MGRSC,
    x=WINDOW[0], y=WINDOW[1], width=WINDOW[2], height=WINDOW[3],
    dates_to_mask=DATES_TO_MASK,
)
batch_custom = sample_to_batch(sample_custom)
batch_custom, y_pred_custom = imputation.impute_sample(batch_custom)

T_out = len(sample_custom["S2_dates"])
print(f"\nSéquence résultante : {T_out} dates (claires uniquement)")
print(f"Dates dans la séquence : {sample_custom['S2_dates']}")
print(f"Prédiction shape : {y_pred_custom.shape}")


plot_seq_RGB_NIR_interactive(
    targets=batch_custom["y"][0],
    inputs=batch_custom["x"][0],
    preds=y_pred_custom[0],
    n_visible=8,
    title=f"Masquage MANUEL dates {DATES_TO_MASK} — {MGRSC} — fenêtre {WINDOW}",
    dates=sample_custom["S2_dates"],
)

Indices masqués : [3, 5, 8, 60, 63, 65]
Dates masquées  : ['2022-09-16', '2022-09-26', '2022-10-11', '2023-07-08', '2023-07-23', '2023-08-02']
(les dates nuageuses seront retirées de la séquence)

Séquence résultante : 48 dates (claires uniquement)
Dates dans la séquence : ['2022-09-01', '2022-09-11', '2022-09-16', '2022-09-21', '2022-09-26', '2022-10-06', '2022-10-11', '2022-10-26', '2022-10-31', '2022-11-05', '2022-11-20', '2022-11-30', '2022-12-05', '2022-12-25', '2023-01-04', '2023-01-19', '2023-01-29', '2023-02-03', '2023-02-08', '2023-02-13', '2023-02-18', '2023-03-05', '2023-03-15', '2023-03-20', '2023-03-25', '2023-03-30', '2023-04-14', '2023-04-19', '2023-05-04', '2023-05-14', '2023-05-24', '2023-06-03', '2023-06-08', '2023-06-23', '2023-06-28', '2023-07-03', '2023-07-08', '2023-07-13', '2023-07-23', '2023-07-28', '2023-08-02', '2023-08-07', '2023-08-12', '2023-08-22', '2023-09-01', '2023-09-06', '2023-09-11', '2023-09-26']
Prédiction shape : torch.Size([1, 48, 10, 128, 128])


In [ ]:
plot_seq_RGB_NIR_interactive(
    targets=batch_custom["y"][0],
    inputs=batch_custom["x"][0],
    preds=y_pred_custom[0],
    n_visible=8,
    title=f"Masquage MANUEL dates {DATES_TO_MASK} — {MGRSC} — fenêtre {WINDOW}",
    dates=sample_custom["S2_dates"],
)